# 🧩 Andamiaje · Proyecto **Centinela** — Fase 1 (Línea base con MLP)
### Redes Neuronales — Deep Learning · Maestría en Ciencia de Datos · Universidad Santo Tomás

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JotaMao1985/Deep_Learning_Usta/blob/main/notebooks/02-scaffold-centinela-fase1.ipynb)

Este cuaderno es un **andamiaje guía** (*scaffold*) para construir la **línea base tabular con MLP** de la Fase 1 del Proyecto Integrador *Centinela*. Trae lista la mecánica de soporte —carga de datos, exploración y funciones de graficado— y deja marcadas con `# TODO (estudiante)` las piezas que constituyen el trabajo evaluable: la **arquitectura del MLP**, el **bucle de entrenamiento** y el **experimento de sobreajuste**.

> **Cómo se usa:** se clona este cuaderno, se ejecutan las celdas de soporte en orden y se completan las celdas marcadas `# TODO (estudiante)`. En Google Colab: *Entorno de ejecución → Ejecutar todas* (las celdas TODO fallarán a propósito hasta que se completen).

> ⚠️ **Andamiaje — NO es la solución.** Este cuaderno **no** contiene un MLP entrenado ni un bucle de entrenamiento resuelto, y usa **datos sintéticos** (`make_moons` / `make_circles`), **no** los datos reales del proyecto. Su propósito es enseñar la *mecánica*; el diseño del modelo, el entrenamiento y el análisis los aporta el estudiante sobre **sus propios datos**.

---
**Contenido**
0. Preparación del entorno
1. Carga de datos sintéticos y partición *train / val / test*
2. Exploración mínima (EDA)
3. `# TODO (estudiante)` · Arquitectura del MLP (≥ 2 capas ocultas)
4. `# TODO (estudiante)` · Pérdida, optimizador y bucle de entrenamiento (50 épocas)
5. Funciones de diagnóstico provistas (curvas de pérdida y matriz de confusión)
6. `# TODO (estudiante)` · Inducir y corregir el sobreajuste con *Dropout*
7. Cierre · Lista de entregables


## 0. Preparación del entorno

Se importan las librerías de soporte (`numpy`, `scikit-learn`, `matplotlib`) y se fija una **semilla** para que los resultados sean reproducibles. La separación de miles se mostrará con punto, según la convención del material del curso.

> 💡 *PyTorch como TODO opcional:* las celdas de soporte de este andamiaje **no** requieren `torch`. La importación de PyTorch se deja como tarea del estudiante en la Sección 3, donde se define el modelo.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 1 · Preparación del entorno
# Qué hace · Importa las librerías de soporte, fija la semilla y configura el estilo de las gráficas.
# Por qué  · Una semilla fija hace reproducibles los resultados; la paleta da identidad USTA.

# En Colab estas librerías ya vienen instaladas. Si faltara alguna, descomentar:
# !pip -q install scikit-learn matplotlib

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

# Reproducibilidad: una misma semilla produce los mismos resultados en cada ejecucion.
SEMILLA = 42
np.random.seed(SEMILLA)

# Paleta USTA para las graficas
USTA_MORADO, USTA_ROSA, USTA_NAVY = "#3D008D", "#ED1E79", "#001A4D"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

# Separador de miles con punto, segun la convencion del material del curso.
def miles(n):
    """Formatea un entero con punto como separador de miles (p. ej. 1.234.567)."""
    return f"{n:,}".replace(",", ".")

print("Entorno de soporte listo · numpy", np.__version__)


## 1. Carga de datos sintéticos y partición *train / val / test*

Para practicar la mecánica **sin adelantar nada del proyecto**, se usa un conjunto **sintético** de clasificación binaria. Por defecto se generan **dos lunas** (`make_moons`): dos clases entrelazadas que ninguna recta separa, con la misma *forma* de dificultad que un problema real no lineal.

> 💡 *Para explorar:* basta cambiar `CONJUNTO = "lunas"` por `CONJUNTO = "circulos"` y volver a ejecutar desde aquí para trabajar con `make_circles`.

> ☕ **Puente con el proyecto:** en el proyecto real estas dos columnas serán **características tabulares** del problema elegido (intensidades de imagen, agregados de clima, etc.) y la etiqueta será el **evento de riesgo**. Aquí son coordenadas sintéticas; el andamiaje es el mismo.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 2 · Datos sinteticos y particion train / val / test
# Qué hace · Genera el conjunto, lo estandariza y lo divide en tres particiones (60/20/20).
# Por qué  · Una particion en tres conjuntos permite entrenar, ajustar y evaluar sin contaminar la prueba.

CONJUNTO = "lunas"   # opciones: "lunas" (make_moons) o "circulos" (make_circles)
N_MUESTRAS = 600
RUIDO = 0.20

if CONJUNTO == "lunas":
    X, y = make_moons(n_samples=N_MUESTRAS, noise=RUIDO, random_state=SEMILLA)
elif CONJUNTO == "circulos":
    X, y = make_circles(n_samples=N_MUESTRAS, noise=RUIDO, factor=0.5, random_state=SEMILLA)
else:
    raise ValueError("CONJUNTO debe ser 'lunas' o 'circulos'.")

# Estandarizar (media 0, desviacion 1) ayuda a que el entrenamiento converja mejor.
X = StandardScaler().fit_transform(X)

# Particion 60 % entrenamiento / 20 % validacion / 20 % prueba.
# Primero se aparta la prueba; luego se divide el resto en entrenamiento y validacion.
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMILLA, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEMILLA, stratify=y_temp)  # 0.25 * 0.80 = 0.20

print(f"Conjunto sintetico: '{CONJUNTO}' · {miles(N_MUESTRAS)} muestras · 2 caracteristicas")
print(f"Entrenamiento: {miles(len(X_train))}  ·  Validacion: {miles(len(X_val))}  ·  Prueba: {miles(len(X_test))}")


## 2. Exploración mínima (EDA)

Antes de modelar conviene **mirar los datos**. Se grafica la dispersión de las dos clases sobre el conjunto de entrenamiento y se revisa el **balance de clases** (cuántos ejemplos hay de cada una), un dato que más adelante condiciona la lectura de la matriz de confusión.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 3 · Exploracion minima (EDA)
# Qué hace · Dibuja la dispersion de las dos clases e informa el balance de clases.
# Por qué  · Ver la forma del problema orienta el diseno del modelo y la lectura posterior de metricas.

fig, ax = plt.subplots(figsize=(5.5, 4.6))
for clase, color, etiqueta in [(0, USTA_NAVY, "Clase 0"), (1, USTA_ROSA, "Clase 1")]:
    mascara = y_train == clase
    ax.scatter(X_train[mascara, 0], X_train[mascara, 1],
               c=color, s=22, edgecolors="white", linewidths=0.4, label=etiqueta)
ax.set_xlabel("Caracteristica 1 (estandarizada)")
ax.set_ylabel("Caracteristica 2 (estandarizada)")
ax.set_title(f"Dispersion de clases · conjunto '{CONJUNTO}' (entrenamiento)")
ax.legend()
plt.tight_layout(); plt.show()

n0 = int((y_train == 0).sum()); n1 = int((y_train == 1).sum())
print(f"Balance de clases en entrenamiento → Clase 0: {miles(n0)}   Clase 1: {miles(n1)}")


## 3. `# TODO (estudiante)` · Arquitectura del MLP

> [!todo] Tarea del estudiante
> Aquí inicia el trabajo evaluable. Se debe **definir la arquitectura** de un perceptrón multicapa para clasificación binaria, según exige la actividad.

**Requisitos mínimos:**

1. Importar PyTorch (`import torch`, `import torch.nn as nn`) y fijar también la semilla de torch (`torch.manual_seed(SEMILLA)`) para reproducibilidad.
2. Definir una **clase** que herede de `nn.Module` con **al menos dos capas ocultas**.
3. Usar una **no linealidad** en las capas ocultas (p. ej. `ReLU`) y una salida adecuada para clasificación **binaria** (p. ej. una unidad con `Sigmoid`, o una unidad lineal si luego se usa `BCEWithLogitsLoss`).
4. **Justificar** en una celda de texto la elección de las funciones de activación, apoyándose en la teoría de no linealidad del Módulo 1.
5. Prever un argumento de **`Dropout`** (probabilidad configurable, por defecto 0) que se usará en la Sección 6.

> 💡 *Pista de forma:* la entrada tiene **2 características** y la salida es **1** (probabilidad de la clase 1). La pérdida `BCELoss` espera las etiquetas con forma `(N, 1)`.

> ⚠️ La celda siguiente está **intencionalmente incompleta**: lanza `NotImplementedError` hasta que se reemplace el cuerpo por la implementación propia.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Definir la arquitectura del MLP
# Reemplazar el cuerpo de la clase por la implementacion propia (>= 2 capas ocultas).
# Eliminar la linea 'raise NotImplementedError(...)' una vez completada.

# import torch                # TODO: descomentar
# import torch.nn as nn       # TODO: descomentar
# torch.manual_seed(SEMILLA)  # TODO: fijar la semilla de torch

class MLP:  # TODO: heredar de nn.Module  ->  class MLP(nn.Module):
    """MLP para clasificacion binaria: 2 -> [ocultas] -> 1.

    TODO (estudiante):
      - definir las capas en __init__ (al menos dos capas ocultas);
      - aplicar una no linealidad (p. ej. ReLU) en las ocultas;
      - elegir la salida adecuada para clasificacion binaria;
      - dejar previsto un argumento p_dropout (por defecto 0.0) para la Seccion 6;
      - implementar forward(self, x) devolviendo la prediccion.
    """
    def __init__(self, n_entrada=2, n_oculta=16, p_dropout=0.0):
        raise NotImplementedError(
            "TODO (estudiante): definir la arquitectura del MLP con >= 2 capas ocultas.")

    def forward(self, x):
        raise NotImplementedError("TODO (estudiante): implementar el forward pass.")


## 4. `# TODO (estudiante)` · Pérdida, optimizador y bucle de entrenamiento

> [!todo] Tarea del estudiante
> Con el modelo ya definido, se debe **configurar el entrenamiento** y escribir el **bucle explícito**, según exige la actividad: por **50 épocas**, registrando la pérdida de entrenamiento y de validación en cada época.

**Los cinco pasos que cada época debe ejecutar, en orden:**

1. **`zero_grad`** — limpiar los gradientes acumulados de la época anterior.
2. **forward** — calcular las predicciones del modelo sobre el conjunto de entrenamiento.
3. **pérdida** — comparar las predicciones con la verdad (p. ej. `BCELoss`).
4. **`backward`** — retropropagar el error (calcular los gradientes).
5. **`step`** — el optimizador actualiza los pesos.

**Indicaciones:**

- Convertir `X_train, y_train, X_val, y_val` a **tensores** de PyTorch (`float32`); las etiquetas con forma `(N, 1)`.
- Elegir una **función de pérdida** para clasificación binaria y un **optimizador** (SGD o Adam) con una **tasa de aprendizaje** inicial.
- Devolver el historial de pérdidas (`train` y `val`) para graficarlo con los helpers de la Sección 5.

> ⚠️ La celda siguiente está **intencionalmente incompleta**. Debe completarse el cuerpo de `entrenar(...)` y la conversión a tensores antes de ejecutarla.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Configurar perdida/optimizador y escribir el bucle de entrenamiento
# Completar la conversion a tensores y el cuerpo de entrenar(); eliminar el raise final.

# --- 1) Conversion a tensores de PyTorch (TODO) -----------------------------
# def a_tensor(Xa, ya):
#     return (torch.tensor(Xa, dtype=torch.float32),
#             torch.tensor(ya, dtype=torch.float32).view(-1, 1))
# X_train_t, y_train_t = a_tensor(X_train, y_train)
# X_val_t,   y_val_t   = a_tensor(X_val,   y_val)

# --- 2) Bucle de entrenamiento explicito (TODO) -----------------------------
def entrenar(modelo, X_tr, y_tr, X_va, y_va, epocas=50, lr=0.05):
    """Entrena el MLP por 'epocas' epocas y devuelve el historial de perdidas.

    TODO (estudiante): implementar dentro del bucle, en este orden:
        (1) zero_grad  (2) forward  (3) perdida  (4) backward  (5) step
    Registrar en cada epoca la perdida de entrenamiento y la de validacion
    y devolver, por ejemplo, un dict {"train": [...], "val": [...]}.
    """
    raise NotImplementedError(
        "TODO (estudiante): escribir el bucle explicito de 50 epocas con los cinco pasos.")

# --- 3) Instanciar el modelo y entrenar (TODO) ------------------------------
# modelo = MLP(n_oculta=16, p_dropout=0.0)
# historial = entrenar(modelo, X_train_t, y_train_t, X_val_t, y_val_t, epocas=50, lr=0.05)


## 5. Funciones de diagnóstico provistas

Estas dos funciones **ya están implementadas** y son las que el estudiante **invoca** para diagnosticar el entrenamiento. No dependen de PyTorch: reciben listas o arreglos de `numpy`, de modo que funcionan con el historial y las predicciones que produzca cualquier implementación del modelo.

- `plot_curvas_perdida(train_losses, val_losses)` — superpone las curvas de pérdida de entrenamiento y validación; la **brecha** entre ambas es la firma del sobreajuste.
- `plot_matriz_confusion(y_true, y_pred)` — dibuja la matriz de confusión, que separa *falsos positivos* (FP) de *falsos negativos* (FN), base de la **auditoría ética** del proyecto.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Celda de soporte 4 · Funciones de diagnostico (FUNCIONALES — listas para usar)
# Qué hace · Define plot_curvas_perdida y plot_matriz_confusion con matplotlib/sklearn.
# Por qué  · El estudiante solo las invoca; no dependen de torch, asi que funcionan con numpy.

def plot_curvas_perdida(train_losses, val_losses, titulo="Curva de perdida"):
    """Grafica la perdida de entrenamiento frente a la de validacion por epoca.

    Parametros
    ----------
    train_losses, val_losses : list | np.ndarray
        Perdida registrada en cada epoca para entrenamiento y validacion.
    """
    train_losses = np.asarray(train_losses, dtype=float)
    val_losses = np.asarray(val_losses, dtype=float)
    epocas = np.arange(1, len(train_losses) + 1)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(epocas, train_losses, color=USTA_MORADO, lw=2, label="Entrenamiento")
    ax.plot(epocas, val_losses, color=USTA_ROSA, lw=2, label="Validacion")
    ax.set_xlabel("Epoca"); ax.set_ylabel("Perdida")
    ax.set_title(titulo); ax.legend()
    plt.tight_layout(); plt.show()


def plot_matriz_confusion(y_true, y_pred, etiquetas=("Clase 0", "Clase 1"),
                          titulo="Matriz de confusion"):
    """Dibuja la matriz de confusion (2x2) con sus conteos anotados.

    Parametros
    ----------
    y_true, y_pred : array-like de 0/1
        Etiquetas verdaderas y predichas (mismo largo).
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.4, 4))
    im = ax.imshow(cm, cmap="Purples")
    ax.set_xticks([0, 1], labels=etiquetas)
    ax.set_yticks([0, 1], labels=etiquetas)
    ax.set_xlabel("Prediccion"); ax.set_ylabel("Valor real")
    ax.set_title(titulo)
    umbral = cm.max() / 2 if cm.max() > 0 else 0.5
    for i in range(2):
        for j in range(2):
            ax.text(j, i, miles(int(cm[i, j])), ha="center", va="center",
                    color="white" if cm[i, j] > umbral else "black", fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.grid(False)
    plt.tight_layout(); plt.show()


# --- Comprobacion rapida con datos de juguete (verifica que los helpers funcionan) ---
_demo_train = np.linspace(0.9, 0.1, 50)
_demo_val = _demo_train + np.linspace(0.0, 0.15, 50)
plot_curvas_perdida(_demo_train, _demo_val, titulo="Demostracion de los helpers (datos de juguete)")
plot_matriz_confusion([0, 0, 1, 1, 0, 1], [0, 1, 1, 1, 0, 0],
                      titulo="Demostracion de los helpers (datos de juguete)")
print("Funciones de diagnostico listas: plot_curvas_perdida(...) y plot_matriz_confusion(...).")


## 6. `# TODO (estudiante)` · Inducir y corregir el sobreajuste con *Dropout*

> [!todo] Tarea del estudiante
> Este es el **desafío** de la actividad. Se debe **inducir el sobreajuste a propósito** y luego **corregirlo**, documentando el cambio con las gráficas.

**Pasos sugeridos:**

1. **Inducir el sobreajuste:** entrenar una red **más grande** (más neuronas y/o más épocas) sobre **pocos datos ruidosos**. Graficar las curvas con `plot_curvas_perdida(...)` y observar cómo la pérdida de validación **se separa** de la de entrenamiento.
2. **Corregirlo:** volver a entrenar una red equivalente añadiendo **`Dropout`** (usando el argumento `p_dropout` previsto en la Sección 3) o ajustando la tasa de aprendizaje. Volver a graficar y **comparar la brecha** *train / val* antes y después.
3. **Evaluar en la prueba:** generar las predicciones sobre el conjunto de **prueba** (`X_test`) y dibujar la matriz de confusión con `plot_matriz_confusion(...)`.
4. **Reflexión ética:** comentar el coste asimétrico de los **falsos positivos** frente a los **falsos negativos** en el contexto del problema elegido (conecta con la auditoría ética del informe).

> ⚠️ La celda siguiente está **vacía a propósito**. El estudiante escribe aquí el experimento de sobreajuste, reutilizando `MLP`, `entrenar(...)` y los helpers de diagnóstico.


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TODO (estudiante) · Inducir el sobreajuste y corregirlo con Dropout
#
# Guia (escribir el codigo debajo de cada paso):
#   1) Generar un subconjunto pequeno y ruidoso, o reutilizar pocas muestras de entrenamiento.
#   2) Entrenar una red grande SIN regularizar  -> plot_curvas_perdida(...)  (se vera la brecha).
#   3) Entrenar una red equivalente CON Dropout  -> plot_curvas_perdida(...)  (brecha menor).
#   4) Evaluar en el conjunto de PRUEBA          -> plot_matriz_confusion(y_test, y_pred).
#   5) Comentar el coste de los falsos positivos frente a los falsos negativos.
#
# (Celda intencionalmente vacia: el experimento lo construye el estudiante.)


## 7. Cierre · Lista de entregables

Una vez completadas las secciones 3, 4 y 6 sobre los **datos propios** del proyecto (no sobre estos datos sintéticos), la Fase 1 se entrega con tres piezas:

- [ ] **Notebook** (`.ipynb`) comentado: carga y EDA de los datos reales, arquitectura del MLP justificada, bucle de entrenamiento explícito por 50 épocas, curvas de pérdida y experimento de sobreajuste/Dropout documentado.
- [ ] **Informe técnico-ético** (PDF, máx. 3 páginas): análisis de las gráficas y la reflexión ética sobre el coste de los falsos positivos para el cliente.
- [ ] **Bitácora de IA**: registro de las interacciones relevantes con la IA generativa —qué se pidió, qué devolvió, qué se **aceptó / modificó / rechazó** y **por qué**—, incluyendo al menos **un caso en que la IA falló** y cómo se corrigió.

> [!warning] Este andamiaje no se entrega tal cual
> Este cuaderno es punto de partida y trabaja con **datos sintéticos**. La entrega evaluable debe aplicarse a los **datos reales** del problema elegido y aprobado, e incluir el informe y la bitácora. Reutilizar este scaffold sin adaptarlo al propio problema **no satisface** la actividad.

### 🔗 Relacionado
- Página principal de la bóveda y mapa del Segundo Cerebro.
- Syllabus del Curso y Ruta de Aprendizaje.
- Documento maestro: **Proyecto Integrador Centinela** (especificación longitudinal de las tres fases).
- **Fase 1 (evaluable):** Actividad 1 · "El Guardián del Cafetal".
- **Material teórico:** Módulo 1 — Capítulos 2 (MLP), 3 (optimización) y 5 (regularización y ética).
- **Uso de IA:** plantilla de la Bitácora de IA para registrar el proceso de la fase.
